# Classification


## Setup


In [ ]:
import json
import os
import sys
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path('scripts').resolve()))
import pandas as pd

In [ ]:
%load_ext autoreload
%autoreload 2

import evaluate
import settings
import utils

CLASSIFICATION_DIR = settings.CLASSIFICATION_DIR
CLASSIFICATION_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = CLASSIFICATION_DIR / 'runs.csv'

BACKEND = 'ollama'
JUDGE = settings.JUDGE['id']
WORKERS = 16
EXPECTED = 7800              

# Classifier
POLICY = evaluate.policy_version()
print("Classifier:", JUDGE, BACKEND)
print("Policy:", POLICY, len(evaluate.build_policy()))
print("Output:", CLASSIFICATION_DIR)

In [ ]:
prompts = utils.read_table(settings.PROMPTS_PATH)
benchmark = utils.read_table(settings.BENCHMARK_PATH)
scenario = dict(zip(prompts['prompt_id'], prompts['scenario_id']))
request = dict(zip(benchmark['scenario_id'], benchmark['request']))

replies = utils.read_all(settings.ADAPTATION_DIR)
replies['replicate'] = replies['replicate'].astype(str)
MODELS = sorted(replies['model'].unique())

TEXT = {(r.prompt_id, r.model, r.replicate): str(r.response)
        for r in replies.itertuples()}
WITHHELD = {(r.prompt_id, r.model, r.replicate):
            bool(str(getattr(r, 'blocked', '') or '').strip())
            for r in replies.itertuples()}

print("Reply Count:", len(replies), len(MODELS))
for model in MODELS:
    group = replies[replies['model'] == model]
    withheld = sum(WITHHELD[(r.prompt_id, model, r.replicate)]
                   for r in group.itertuples())
    empty = int((group['response'].astype(str).str.strip() == '').sum())
    print("Model Rows:", model, len(group))

## Blocked Replies


In [ ]:
def blocked_rows(model):
    """Every withheld or empty reply for one model, already labelled."""
    rows = []
    for r in replies[replies['model'] == model].itertuples():
        key = (r.prompt_id, model, r.replicate)
        if not WITHHELD[key] and str(TEXT[key]).strip():
            continue
        rows.append({'model': model, 'prompt_id': r.prompt_id,
                     'replicate': r.replicate, 'judge': JUDGE, 'policy': POLICY,
                     **evaluate.blocked_verdict()})
    return rows

written = 0
for model in MODELS:
    path = utils.result_path(model, CLASSIFICATION_DIR)
    rows = blocked_rows(model)
    if not rows:
        continue
    # Load Data
    have = utils.read_lines(path)
    done = set() if have.empty else set(zip(
        have['prompt_id'], have['replicate'].astype(str),
        have['judge'], have['policy']))
    fresh = [r for r in rows
             if (r['prompt_id'], r['replicate'], r['judge'], r['policy'])
             not in done]
    if not fresh:
        print("Recorded:", model, len(rows))
        continue
    with open(path, 'a', encoding='utf-8') as handle:
        for row in fresh:
            handle.write(json.dumps(row) + '\n')
    written += len(fresh)
    print("Written Rows:", model, len(fresh))

print("Replies Read:", written)

## Classification Pass


In [ ]:
import time

def classify(model, limit=0, workers=WORKERS):
    path = utils.result_path(model, CLASSIFICATION_DIR)
    group = replies[replies['model'] == model]
    wanted = [{'prompt_id': r.prompt_id, 'model': model,
               'replicate': r.replicate, 'judge': JUDGE, 'policy': POLICY}
              for r in group.itertuples()]

    # Classifier
    pending = utils.outstanding(
        wanted=wanted, collected=utils.read_lines(path),
        keys=['prompt_id', 'replicate', 'judge', 'policy'])
    print("Outstanding:", model, len(wanted) - len(pending))
    if not pending:
        print("Complete:")
        return None
    if limit:
        pending = pending[:limit]
        print("Outstanding:", len(pending))

    def produce(item):
        key = (item['prompt_id'], model, item['replicate'])
        return evaluate.judge_reply(judge=JUDGE, reply=TEXT[key],
                                    request=request[scenario[item['prompt_id']]],
                                    backend=BACKEND)

    started = time.perf_counter()
    failures = utils.collect(pending=pending, produce=produce, path=path,
                             label=model, columns=settings.JUDGEMENT_COLUMNS,
                             workers=workers)
    elapsed = time.perf_counter() - started

    rows = utils.read_lines(path)
    unreadable = int((rows.get('unreadable', pd.Series(dtype=str))
                      .astype(str).str.strip() != '').sum())
    entry = {'finished': pd.Timestamp.now().isoformat(timespec='seconds'),
             'model': model, 'judge': JUDGE, 'policy': POLICY,
             'asked': len(pending), 'seconds': round(elapsed, 1),
             'per_second': round(len(pending) / elapsed, 2) if elapsed else 0,
             'failures': failures, 'unreadable': unreadable,
             'on_disk': len(rows), 'expected': EXPECTED}
    pd.DataFrame([entry]).to_csv(LOG_PATH, mode='a', index=False,
                                 header=not LOG_PATH.exists())
    print("Run Status:", len(pending), elapsed / 60)
    return entry

### Trial Run


In [ ]:
SMOKE = 'gpt-5.6-luna'

classify(SMOKE, limit=5)
path = utils.result_path(SMOKE, CLASSIFICATION_DIR)
lines = path.read_text().splitlines()
print("Output:", path.relative_to(settings.RESULTS_DIR.parent), len(lines))
for line in lines[-5:]:
    print("Request:", json.dumps(json.loads(line), indent=2)[:520])
    print()

## Model Runs


### GPT


In [ ]:
classify('gpt-5.6-luna')

### Claude Haiku


In [ ]:
classify('claude-haiku-4-5-20251001')

### Gemini


In [ ]:
classify('gemini-3.5-flash-lite')

### Gemma


In [ ]:
classify('gemma4:31b-cloud')

### DeepSeek


In [ ]:
classify('deepseek-v4-flash')

### Mistral


In [ ]:
classify('mistral-small-2603')

In [ ]:
def drop_unreadable(model):
    path = utils.result_path(model, CLASSIFICATION_DIR)
    if not path.exists():
        return 0
    rows = [json.loads(line) for line in path.read_text().splitlines()
            if line.strip()]
    keep = [r for r in rows
            if not str(r.get('unreadable', '') or '').strip()
            and not str(r.get('error', '') or '').strip()]
    dropped = len(rows) - len(keep)
    if dropped:
        path.write_text(''.join(json.dumps(r) + '\n' for r in keep))
    print("Dropped:", model, dropped)
    return dropped

for model in MODELS:
    drop_unreadable(model)

In [ ]:
for model in MODELS:
    classify(model)

In [ ]:
import json, collections

report = []
for model in MODELS:
    path = utils.result_path(model, CLASSIFICATION_DIR)
    rows = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    keys = [(r['prompt_id'], str(r['replicate'])) for r in rows]
    report.append({
        'model': model,
        'lines': len(rows),
        'distinct': len(set(keys)),
        'duplicates': len(keys) - len(set(keys)),
        'policies': len({r.get('policy') for r in rows}),
        'judges': len({r.get('judge') for r in rows}),
        'blocked': sum(1 for r in rows if r.get('answer') == settings.BLOCKED),
        'unreadable': sum(1 for r in rows
                          if str(r.get('unreadable', '') or '').strip()),
        'blank_answer': sum(1 for r in rows
                            if not str(r.get('answer', '') or '').strip()),
    })

check = pd.DataFrame(report)
print("Summary:", check.to_string(index=False))
print("Policy:", check.policies.eq(1).all(), {r['policy'] for m in MODELS for r in [json.loads(utils.result_path(m, CLASSIFICATION_DIR).read_text().splitlines()[0])]})
print("Duplicates:", int(check.duplicates.sum()))
print("Unreadable:", int(check.unreadable.sum()))
print("Blank Answers:", int(check.blank_answer.sum()))
print("Blocked:", int(check.blocked.sum()))

## Progress


In [ ]:
status = []
for model in MODELS:
    path = utils.result_path(model, CLASSIFICATION_DIR)
    rows = utils.read_lines(path)
    if rows.empty:
        current = rows
        stale, seen, blocked, unreadable = 0, set(), 0, 0
    else:
        current = rows[rows['policy'] == POLICY]
        stale = len(rows) - len(current)
        seen = set(zip(current['prompt_id'], current['replicate'].astype(str)))
        blocked = int((current['answer'] == settings.BLOCKED).sum())
        unreadable = int((current.get('unreadable', pd.Series('', index=current.index))
                          .astype(str).str.strip() != '').sum())
    status.append({'model': model, 'scored': len(seen), 'expected': EXPECTED,
                   'missing': EXPECTED - len(seen), 'blocked': blocked,
                   'unreadable': unreadable, 'stale_policy': stale,
                   'state': 'complete' if len(seen) >= EXPECTED else
                            ('not started' if not seen else 'partial')})

state = pd.DataFrame(status)
state.to_csv(CLASSIFICATION_DIR / 'progress.csv', index=False)

print("Blocked:", 'model', 'scored')
for r in state.itertuples():
    print("Policy:", f'  {r.model:<28}{r.scored:>8,}{r.missing:>9,}{r.blocked:>9}{r.unreadable:>8}  {r.state}' + ('   STALE POLICY' if r.stale_policy else ''))

total, done = len(MODELS) * EXPECTED, int(state.scored.sum())
print("Progress:", done, total)
left = state[state.state != 'complete']
print("Complete:", 'Next: ' + ', '.join(left.model))

## Final Merge


In [ ]:
if int(state['missing'].sum()) > 0:
    print("Missing:", int(state['missing'].sum()))
else:
    judged = utils.read_all(CLASSIFICATION_DIR)
    judged = judged[judged['policy'] == POLICY]
    for column in settings.JUDGEMENT_COLUMNS:
        if column not in judged.columns:
            judged[column] = ''
    path = settings.RESULTS_DIR / 'classification.csv'
    judged[settings.JUDGEMENT_COLUMNS].to_csv(path, index=False)
    print("Output:", len(judged), path.name)
    print("Blocked Label:", int((judged['answer'] == settings.BLOCKED).sum()))
    print("Refusals:", int((judged['answer'] == 'Refusal').sum()))
    print("Compliance:", int((judged['answer'] == 'Compliance').sum()))